# Closed-to-open transition trajectories

Run notebook 03 first.


In [26]:
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import xarray as xr

SOURCE = Path("../../resources_mesoscale_structure/CO_transitions_EC_GOES.nc")
raw = xr.open_dataset(SOURCE).load()

OUTPUT_ZARR = Path("output/earthcare-marine-stratocumulus-transitions/co_transitions_ec_goes/earthcode_co_transitions_ec_goes.zarr")
OUTPUT_ZARR.parent.mkdir(parents=True, exist_ok=True)


## Read data and scene attributes


In [27]:
# Scene attributes from notebook 03.
ds = raw.copy()
scenes = xr.open_zarr("output/earthcare-marine-stratocumulus-transitions/acmcap_judge_stcu/earthcode_acmcap_judge_stcu.zarr", consolidated=True)
for name in ds.variables:
    if name in scenes:
        for key, value in scenes[name].attrs.items():
            ds[name].attrs[key] = value
ds


<xarray.Dataset> Size: 84MB
Dimensions:                               (time_startpoint: 12920, timestep: 43)
Coordinates:
  * time_startpoint                       (time_startpoint) datetime64[ns] 103kB ...
  * timestep                              (timestep) int64 344B -42 -40 ... 42
    orbit                                 (time_startpoint) <U6 310kB '01009E...
Data variables: (12/136)
    lat_traj                              (time_startpoint, timestep) float64 4MB ...
    lon_traj                              (time_startpoint, timestep) float64 4MB ...
    plev_traj                             (time_startpoint, timestep) float64 4MB ...
    datetime_traj                         (time_startpoint, timestep) datetime64[ns] 4MB ...
    goes_cats                             (time_startpoint, timestep) float64 4MB ...
    goes_cert                             (time_startpoint, timestep) float64 4MB ...
    ...                                    ...
    cth_m_cv                              (time_startpoint) float64 103kB 0.0...
    lwp_cv                                (time_startpoint) float64 103kB 0.5...
    ER_cv                                 (time_startpoint) float64 103kB 0.3...
    Nd_cv                                 (time_startpoint) float64 103kB 0.3...
    rwp_cv                                (time_startpoint) float64 103kB nan...
    rain_median_volume_diameter_cv        (time_startpoint) float64 103kB nan...

## Fix trajectory times

Restore invalid times to `NaT`; encode as nanoseconds since 1970-01-01 UTC.


In [28]:
# Missing datetimes were interpolated as integers in the original processing.
time = ds.datetime_traj
bad_time = time.notnull() & (time != time.sel(timestep=0, drop=True) + ds.timestep.astype("timedelta64[h]"))

ds["datetime_traj"] = time.where(~bad_time)
ds.datetime_traj.encoding = dict(
    time.encoding, units="nanoseconds since 1970-01-01", _FillValue=np.iinfo("int64").min
)
f"Restored {int(bad_time.sum()):,} invalid timestamps to NaT"

'Restored 23,486 invalid timestamps to NaT'

## Source attributes


In [29]:
# Original GOES-16 samples on public Amazon S3.
!curl -L --fail --create-dirs -o downloads/goes_c14.nc https://noaa-goes16.s3.amazonaws.com/ABI-L1b-RadF/2024/001/00/OR_ABI-L1b-RadF-M6C14_G16_s20240010000205_e20240010009513_c20240010009574.nc
!curl -L --fail --create-dirs -o downloads/goes_cod.nc https://noaa-goes16.s3.amazonaws.com/ABI-L2-COD2KMF/2024/001/00/OR_ABI-L2-COD2KMF-M6_G16_s20240010010205_e20240010019513_c20240010025551.nc

goes_c14 = xr.open_dataset("downloads/goes_c14.nc", engine="h5netcdf")
goes_cod = xr.open_dataset("downloads/goes_cod.nc", engine="h5netcdf")


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 24.3M  100 24.3M    0     0  3323k      0  0:00:07  0:00:07 --:--:-- 3813k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 19.4M  100 19.4M    0     0  4041k      0  0:00:05  0:00:04  0:00:01 3938k79k      0  0:00:06  0:00:02  0:00:04 2878k 0  0:00:04  0:00:04 --:--:-- 4548k


In [ ]:
# EarthCARE source metadata already comes from the mission files in notebook 03.
ds["time_startpoint"].attrs["source_long_name"] = scenes.time_mid.attrs["long_name"]
ds["time_startpoint"].attrs["source_units"] = scenes.time_mid.attrs["source_units"]
ds["time_startpoint"].attrs["standard_name"] = "time"
ds["time_startpoint"].attrs["source_product"] = "EarthCARE track"
ds["time_startpoint"].attrs["source_variable"] = "time"
ds["time_startpoint"].attrs["long_name"] = "Start times of trajectories"
ds["time_startpoint"].attrs["description"] = "Start times of trajectories, which are the times of the EarthCARE overpass (connected to EarthCARE overpass locations). The trajectories are computed for 42 hours backwards and forwards from these times."
ds["timestep"].attrs["units"] = "1"
ds["timestep"].attrs["time_offset_units"] = "hours"
ds["timestep"].attrs["comment"] = "Negative values are backward trajectories; positive values are forward trajectories."
ds["timestep"].attrs["long_name"] = "trajectory time offset from start point"
ds["datetime_traj"].attrs["long_name"] = "time along wind trajectory"
ds["datetime_traj"].attrs["standard_name"] = "time"
ds["datetime_traj"].attrs["source_product"] = "wind trajectory calculation"
ds["datetime_traj"].attrs["description"] = 'UTC datetime along the trajectory. Invalid dates produced by interpolating missing datetimes as integers are restored to NaT; valid dates are unchanged.'
for axis, coordinate in {"lat": "latitude", "lon": "longitude"}.items():
    attrs = scenes[f"{axis}_mid"].attrs
    ds[f"{axis}_traj"].attrs["long_name"] = f"{coordinate} along wind trajectory"
    ds[f"{axis}_traj"].attrs["standard_name"] = coordinate
    ds[f"{axis}_traj"].attrs["units"] = attrs["units"]
    ds[f"{axis}_traj"].attrs["source_product"] = "EarthCARE track and wind trajectory calculation"
    ds[f"{axis}_traj"].attrs["source_variable"] = coordinate
    ds[f"{axis}_traj"].attrs["source_long_name"] = attrs["long_name"]
    ds[f"{axis}_traj"].attrs["source_units"] = attrs["units"]
ds["plev_traj"].attrs["long_name"] = "pressure level along wind trajectory"
ds["plev_traj"].attrs["standard_name"] = "air_pressure"
ds["plev_traj"].attrs["units"] = "hPa"
ds["plev_traj"].attrs["source_product"] = "wind trajectory calculation"

ds["goes_cat_at_ec_overpass"].attrs["long_name"] = "GOES cloud category at EarthCARE overpass (timestep=0)"
ds["goes_cat_at_ec_overpass"].attrs["source_product"] = "GOES neural-network classifier"
ds["goes_cat_at_ec_overpass"].attrs["units"] = "1"
ds["goes_cat_at_ec_overpass"].attrs["flag_values"] = [0, 1, 2, 3, 4, 5]
ds["goes_cat_at_ec_overpass"].attrs["flag_meanings"] = "solid_stratus closed_mcc open_mcc disorganized_mcc clustered_cu suppressed_cu"
for key, value in ds["goes_cat_at_ec_overpass"].attrs.items():
    ds["goes_cats"].attrs[key] = value
ds["goes_cats"].attrs["long_name"] = "GOES neural-network mesoscale cloud morphology class"
ds["goes_cert"].attrs["long_name"] = "GOES neural-network morphology classification confidence"
ds["goes_cert"].attrs["units"] = "1"
ds["goes_cert"].attrs["valid_min"] = 0.0
ds["goes_cert"].attrs["valid_max"] = 1.0
ds["goes_cert"].attrs["source_product"] = "GOES neural-network classifier"

SOURCE_ATTRS = ["long_name", "standard_name", "units", "grid_mapping", "cell_methods", "resolution"]
for name, threshold in {"goes_cloud_cover": 0, "goes_ot_gt_1": 1, "goes_ot_gt_5": 5}.items():
    for key in SOURCE_ATTRS:
        ds[name].attrs[f"source_{key}"] = goes_cod.COD.attrs[key]
    ds[name].attrs["long_name"] = f"fraction of GOES cloud optical depth pixels greater than {threshold}"
    ds[name].attrs["units"] = "1"
    ds[name].attrs["threshold"] = f"COD > {threshold}"
    ds[name].attrs["cell_methods"] = "GOES scene: fraction"
    ds[name].attrs["source_product"] = "GOES ABI L2+ COD2KMF"
    ds[name].attrs["source_variable"] = "COD"
for axis in ["x", "y"]:
    for key in ["long_name", "standard_name", "units", "axis"]:
        ds[f"goes_{axis}_idx"].attrs[f"source_{key}"] = goes_c14[axis].attrs[key]
    ds[f"goes_{axis}_idx"].attrs["long_name"] = f"GOES preprocessing window center {axis} index"
    ds[f"goes_{axis}_idx"].attrs["units"] = "1"
    ds[f"goes_{axis}_idx"].attrs["source_coordinate"] = axis
    ds[f"goes_{axis}_idx"].attrs["grid_mapping"] = "goes_imager_projection"
    ds[f"goes_{axis}_idx"].attrs["scene_window_pixels"] = 128
    ds[f"goes_{axis}_idx"].attrs["goes_grid"] = "upsampled cropped GOES grid"
for key in SOURCE_ATTRS:
    ds["goes_BT14"].attrs[f"source_{key}"] = goes_c14.Rad.attrs[key]
ds["goes_BT14"].attrs["long_name"] = "mean GOES ABI channel 14 brightness temperature"
ds["goes_BT14"].attrs["standard_name"] = "brightness_temperature"
ds["goes_BT14"].attrs["units"] = "K"
ds["goes_BT14"].attrs["cell_methods"] = "GOES scene: mean where ice pixel fraction is below 0.1"
ds["goes_BT14"].attrs["source_product"] = "GOES ABI L1b RadF"
ds["goes_BT14"].attrs["source_variable"] = "Rad"
ds["goes_BT14"].attrs["source_channel"] = 14
ds["goes_BT14"].attrs["source_calibration"] = "radiance converted to brightness temperature with GOES Planck coefficients"

# These fields describe the 64-pixel BT14/COD averaging window.
for axis, coordinate, units in [("lat", "latitude", "degrees_north"), ("lon", "longitude", "degrees_east")]:
    ds[f"goes_{axis}_mid"].attrs["long_name"] = f"{coordinate} at GOES BT14 averaging window center"
    ds[f"goes_{axis}_mid"].attrs["standard_name"] = coordinate
    ds[f"goes_{axis}_mid"].attrs["units"] = units
ds["goes_ice_flag"].attrs["long_name"] = "fraction of pixels with GOES ABI channel 14 brightness temperature below 273 K in the BT14 averaging window"
ds["goes_ice_flag"].attrs["units"] = "1"
ds["goes_ice_flag"].attrs["source_product"] = "GOES ABI L1b RadF"
ds["goes_ice_flag"].attrs["source_variable"] = "BT14"
ds["goes_ice_flag"].attrs["cell_methods"] = "GOES scene: fraction"

for name in ["goes_cloud_cover", "goes_ot_gt_1", "goes_ot_gt_5", "goes_BT14", "goes_ice_flag", "goes_lat_mid", "goes_lon_mid"]:
    ds[name].attrs["grid_mapping"] = "goes_imager_projection"
    ds[name].attrs["scene_window_pixels"] = 64
    ds[name].attrs["goes_grid"] = "GOES ABI grid used for BT14/COD scene summaries"


## Transition metadata


In [31]:
ds.attrs['title'] = 'Wind trajectories with GOES cloud morphology and EarthCARE microphysics for closed-to-open cell transitions'
ds.attrs['summary'] = "This dataset contains EarthCARE cloud properties (microphysical variables and cloud morphology) along with ERA-5 driven wind trajectories and cloud properties from GOES/ABI along these trajectories (including neural network cloud morphology labels). The startpoints for the trajectories are EarthCARE overpasses over the southeast Pacific stratocumulus region. The trajectories are computed for 42 hours backwards and forwards from the EarthCARE overpass times. The cloud morphology and other imager (EarthCARE/MSI and GOES/ABI) variables are computed for 128km x 128km image chips (= scenes); the cloud microphysical variables from EarthCARE's active instruments are computed for 128 km along-track scenes (= 1D). The dataset also contains a variable indicating whether a closed-to-open cell transition was detected along the trajectory, and if so, the time of the transition."
ds.attrs['keywords'] = 'EarthCARE, GOES, stratocumulus, transitions, cloud microphysics'
ds.attrs['license'] = 'CC-BY-4.0'

ds["goes_cats_original"].attrs["long_name"] = "GOES neural-network mesoscale cloud morphology class before manual correction"

ds['transition_flag'].attrs["flag_values"] = [0, 1]
ds['transition_flag'].attrs["flag_meanings"] = 'no_transition transition_detected'
ds["transition_flag"].attrs["long_name"] = "Closed-to-open cell transition flag"
ds["transition_flag"].attrs["description"] = "Flag indicating whether a closed-to-open cell transition was detected along the trajectory. 0 = no transition detected, 1 = transition detected."

ds["transition_flag"].attrs.pop("comment",None)

ds["transition_timestep"].attrs["long_name"] = "timestep of cloud transition"
ds["transition_timestep"].attrs["description"] = "Timestep where transition from closed to open cells occurs (timestep of first open cell point)"

ds['transition_datetime'].attrs["standard_name"] = 'time'
ds["transition_datetime"].attrs["long_name"] = "Time of closed-to-open cell transition"
ds["transition_datetime"].attrs["description"] = "Mean datetime at transition point (mean between last detected closed cell and first detected open cell along the trajectory)."

ds["transition_time_diff"].attrs["long_name"] = "Time difference between EarthCARE overpass and transition"
ds["transition_time_diff"].attrs["units"] = "h"
ds["transition_time_diff"].attrs["description"] = "Time difference from timestep=0 (EarthCARE overpass) to time of the closed-to-open cell transition (positive means transition is before overpass, negative means transition is after overpass)."
ds["transition_time_diff"].attrs["comment"] = ""

ds['transition_uncertainty'].attrs["long_name"] = 'uncertainty of cloud transition timing'
ds['transition_uncertainty'].attrs["units"] = 'h'
ds["transition_uncertainty"].attrs["description"] = "Uncertainty in transition time (half the time between last closed and first open cell as detected by the GOES neural-network classifier)."

ds['rain_fraction_no_zero'].attrs["long_name"] = 'rain fraction with zero values masked'
ds['rain_fraction_no_zero'].attrs["units"] = '1'

for base, description in {
    'cbh_m': 'cloud base height',
    'cth_m': 'cloud top height',
    'lwp': 'liquid water path',
    'ER': 'cloud droplet effective radius',
    'Nd': 'cloud droplet number concentration',
    'rwp': 'rain water path',
    'rain_median_volume_diameter': 'rain median volume diameter',
}.items():
    if base + '_cv' in ds:
        ds[base + '_cv'].attrs["long_name"] = f'coefficient of variation of {description}'
        ds[base + '_cv'].attrs["units"] = '1'
        ds[base + '_cv'].attrs["comment"] = f'{base}_std / {base}'


## Export


In [ ]:
ds.attrs["date_created"] = datetime.now(timezone.utc).isoformat()
for variable in ds.variables.values():
    variable.encoding["chunks"] = variable.shape

ds.to_zarr(OUTPUT_ZARR, mode="w", consolidated=True, zarr_format=2)
OUTPUT_ZARR


PosixPath('output/earthcare-marine-stratocumulus-transitions/co_transitions_ec_goes/earthcode_co_transitions_ec_goes.zarr')

## Check source values


In [33]:
# Skip original values at bad_time; require NaT there instead.
original = xr.open_dataset(SOURCE)
saved = xr.open_zarr(OUTPUT_ZARR, consolidated=True)
xr.testing.assert_equal(saved.drop_vars("datetime_traj"), original.drop_vars("datetime_traj"))
xr.testing.assert_equal(saved.datetime_traj, original.datetime_traj.where(~bad_time))


## Check EarthCODE


In [35]:
cloud = xr.open_zarr(
    'https://s3.waw4-1.cloudferro.com/EarthCODE/OSCAssets/earthcare-marine-stratocumulus-transitions/co_transitions_ec_goes/earthcode_co_transitions_ec_goes.zarr',
    consolidated=True,
)
# Skip the new creation date, summary spacing and added descriptions/units.
for dataset in (saved, cloud):
    dataset.attrs.pop("date_created", None)
    dataset.attrs["summary"] = dataset.attrs["summary"].replace(" ", "")
    for name in ("lat_mid", "lon_mid"):
        dataset[name].attrs.pop("description", None)
    dataset["transition_uncertainty"].attrs.pop("units", None)
# Skip datetime_traj: its dates and metadata were corrected; checked against the source above.
xr.testing.assert_identical(saved.drop_vars("datetime_traj"), cloud.drop_vars("datetime_traj"))
